# Lambda Dafne + MedSAM Thigh Segmentation

Runs the Dafne Thigh model slice-by-slice, then refines each muscle mask with MedSAM using the Dafne bounding box as the prompt. No manual point picking required.

MedSAM embedding is computed **once per slice** and reused for all muscles.

## Before running

Install dependencies (once per instance):
```bash
pip install dafne-dl SimpleITK scikit-image
pip install git+https://github.com/facebookresearch/segment-anything.git
```

**Upload Dafne model:**
```bash
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/dafne_thigh_results/model_used/ \
  ubuntu@129.80.59.179:~/dafne_model/
```

**Upload MedSAM checkpoint:**
```bash
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  "/tmp/docker-desktop-root/run/desktop/mnt/host/c/Users/docto/AppData/Local/Dafne-imaging/Dafne/models/medsam_vit_b.pth" \
  ubuntu@129.80.59.179:~/medsam_vit_b.pth
```

**Upload data** (if not already present):
```bash
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/myosegmenTUM \
  ubuntu@129.80.59.179:~/
```

## Download results when done
```bash
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  ubuntu@129.80.59.179:~/dafne_medsam_results/ \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/dafne_medsam_results/
```

**Terminate the instance when done.**

In [ ]:
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + list(args))

pip('dafne-dl', 'SimpleITK', 'scikit-image')
pip('git+https://github.com/facebookresearch/segment-anything.git')

In [ ]:
import os
os.environ.pop('MPLBACKEND', None)

import glob
import numpy as np
import SimpleITK as sitk
import torch
import torch.nn.functional as F
from skimage import transform
from dafne_dl import DynamicDLModel
from segment_anything import sam_model_registry

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
SAM_DEVICE = 'cpu'   # MedSAM encoder is memory-heavy; keep it on CPU
MODEL_PATH = os.path.expanduser('~/dafne_model/Thigh_1774532147.model')
IMAGE_GLOB = os.path.expanduser('~/myosegmenTUM/*/ImageData/*FATFRACTION/*FATFRACTION_stack*.nii')
OUTPUT_DIR = os.path.expanduser('~/dafne_medsam_results')
SAM_CKPT   = os.path.expanduser('~/medsam_vit_b.pth')

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Dafne device:', DEVICE)
print('SAM device:  ', SAM_DEVICE)

In [ ]:
def enlarge_bounding_box(mask, margin=5):
    rows = np.any(mask, axis=1)
    cols = np.any(mask, axis=0)
    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]
    H, W = mask.shape
    return np.array([
        max(0, cmin - margin),
        max(0, rmin - margin),
        min(W - 1, cmax + margin),
        min(H - 1, rmax + margin),
    ], dtype=float)

def medsam_inference(medsam_model, img_embed, box_1024, H, W):
    box_torch = torch.as_tensor(box_1024, dtype=torch.float, device=img_embed.device)
    if box_torch.ndim == 2:
        box_torch = box_torch[:, None, :]
    sparse_embeddings, dense_embeddings = medsam_model.prompt_encoder(
        points=None, boxes=box_torch, masks=None,
    )
    low_res_logits, _ = medsam_model.mask_decoder(
        image_embeddings=img_embed,
        image_pe=medsam_model.prompt_encoder.get_dense_pe(),
        sparse_prompt_embeddings=sparse_embeddings,
        dense_prompt_embeddings=dense_embeddings,
        multimask_output=False,
    )
    low_res_pred = F.interpolate(
        torch.sigmoid(low_res_logits), size=(H, W), mode='bilinear', align_corners=False
    )
    return (low_res_pred.squeeze().detach().cpu().numpy() > 0.5).astype(np.uint8)

In [ ]:
dafne_model = DynamicDLModel.Load(open(MODEL_PATH, 'rb'))
print('Dafne model loaded:', MODEL_PATH)

In [ ]:
if not os.path.exists(SAM_CKPT):
    raise FileNotFoundError(
        f'MedSAM checkpoint not found at {SAM_CKPT}\n'
        'Upload it with the rsync command in the markdown cell above.'
    )

sam_model = sam_model_registry['vit_b'](checkpoint=SAM_CKPT)
sam_model.to(device=SAM_DEVICE)
sam_model.eval()
print('MedSAM loaded on', SAM_DEVICE)

In [ ]:
image_files = sorted(glob.glob(IMAGE_GLOB))
print(f'Found {len(image_files)} images:')
for p in image_files:
    print(' ', p)

In [ ]:
for nii_path in image_files:
    stem     = os.path.splitext(os.path.basename(nii_path))[0]
    out_path = os.path.join(OUTPUT_DIR, f'{stem}_dafne_medsam.npz')

    if os.path.exists(out_path):
        print(f'Skipping (already done): {out_path}')
        continue

    print(f'\nProcessing: {nii_path}')
    img_sitk   = sitk.ReadImage(nii_path)
    img_array  = sitk.GetArrayFromImage(img_sitk).astype(float)  # (slices, H, W)
    spacing    = img_sitk.GetSpacing()
    resolution = [spacing[0], spacing[1]]
    H, W = img_array.shape[1], img_array.shape[2]
    print(f'  Shape: {img_array.shape}  Resolution: {resolution}')

    all_masks = {}  # {muscle_name: 3D uint8 array}

    for slice_idx in range(img_array.shape[0]):
        slice_2d = img_array[slice_idx]

        # Dafne segmentation
        dafne_out = dafne_model({
            'image':            slice_2d,
            'resolution':       resolution,
            'split_laterality': True,
            'classification':   'Thigh',
        })

        # MedSAM embedding — once per slice, reused for all muscles
        img_norm   = slice_2d * 255.0 / (slice_2d.max() + 1e-8)
        img_3c     = np.repeat(img_norm[:, :, None], 3, axis=-1)
        img_1024   = transform.resize(
            img_3c, (1024, 1024), order=3, preserve_range=True, anti_aliasing=True
        ).astype(np.uint8)
        img_1024   = (img_1024 - img_1024.min()) / np.clip(
            img_1024.max() - img_1024.min(), a_min=1e-8, a_max=None
        )
        img_tensor = torch.tensor(img_1024).float().permute(2, 0, 1).unsqueeze(0).to(SAM_DEVICE)
        with torch.no_grad():
            image_embedding = sam_model.image_encoder(img_tensor)
        del img_tensor

        # refine each muscle mask with MedSAM
        for muscle_name, mask in dafne_out.items():
            mask_arr = np.asarray(mask, dtype=np.uint8)

            if mask_arr.any():
                bbox     = enlarge_bounding_box(mask_arr)         # [min_col, min_row, max_col, max_row]
                box_1024 = bbox / np.array([W, H, W, H]) * 1024
                box_1024 = box_1024[None, None, :]                # (1, 1, 4)
                refined  = medsam_inference(sam_model, image_embedding, box_1024, H, W)
            else:
                refined = mask_arr

            if muscle_name not in all_masks:
                all_masks[muscle_name] = np.zeros(img_array.shape, dtype=np.uint8)
            all_masks[muscle_name][slice_idx] = refined.astype(np.uint8)

        del image_embedding

        if (slice_idx + 1) % 5 == 0 or slice_idx == img_array.shape[0] - 1:
            print(f'  slice {slice_idx + 1}/{img_array.shape[0]} done')

    np.savez_compressed(out_path, **all_masks)
    print(f'  Saved → {out_path}')
    print(f'  Muscles: {list(all_masks.keys())}')

print('\nAll done.')

In [ ]:
results = sorted(glob.glob(os.path.join(OUTPUT_DIR, '*.npz')))
if results:
    sample = np.load(results[0])
    print('Sample file:', results[0])
    for name in sample.files:
        arr = sample[name]
        print(f'  {name}: shape={arr.shape}  positive voxels={arr.sum()}')